# Pyr/CA3 Mesh Download And Decimation By List

This notebook generalizes the proven single-root mesh workflow from `02_pyr_mesh_download.ipynb` to a root-ID batch. It uses flat `seg_m195` as the primary mesh source and lazily uses the CAVE-advertised Graphene source only when the flat source raises `MeshDecodeError` with `Manifest not found`. Existing outputs are validated and reused, new H5/PLY outputs are written through validated temporary files, results are logged incrementally, per-root failures are isolated, and final copy/paste-ready ID lists are printed. `missing_manifest_ids` means IDs still unavailable after the Graphene fallback attempt.


## Parameters

A list containing one root ID should behave like the proven one-off workflow. Set `overwrite_existing = True` manually only when replacing existing H5 or PLY files is intentional.

In [ ]:
root_ids = [648518346458354357]

materialization_version = 195
dec_prcnt = 95
overwrite_existing = False
show_full_path = False

segmentation_source_gs = "precomputed://gs://zheng_mouse_hippocampus_production/v2/seg_m195/"
segmentation_source_https = "precomputed://https://storage.googleapis.com/zheng_mouse_hippocampus_production/v2/seg_m195/"

print(f"root_ids: {[str(root_id) for root_id in root_ids]}")
print(f"materialization_version: {materialization_version}")
print(f"dec_prcnt: {dec_prcnt}")
print(f"overwrite_existing: {overwrite_existing}")

root_ids: ['648518346458354357']
materialization_version: 195
dec_prcnt: 95
overwrite_existing: False


## Environment Diagnostic

Check required packages in the existing environment. Do not install anything here.

In [ ]:
# import importlib
# from importlib import metadata

# required_packages = {
#     "meshparty": "meshparty",
#     "cloudvolume": "cloud-volume",
#     "h5py": "h5py",
#     "numpy": "numpy",
#     "pyvista": "pyvista",
#     "vtk": "vtk",
#     "pandas": "pandas",
#     "caveclient": "caveclient",
#     "dotenv": "python-dotenv",
#     "psutil": "psutil",
# }

# missing_packages = []
# for import_name, distribution_name in required_packages.items():
#     try:
#         importlib.import_module(import_name)
#         try:
#             version = metadata.version(distribution_name)
#         except metadata.PackageNotFoundError:
#             version = "installed, version unavailable"
#         print(f"{import_name}: {version}")
#     except ImportError:
#         missing_packages.append(import_name)
#         print(f"{import_name}: MISSING")

# if missing_packages:
#     raise RuntimeError(
#         "Required package(s) missing from this environment: "
#         + ", ".join(missing_packages)
#         + ". Install them in the notebook kernel environment before continuing."
#     )

## Shared Imports

Import the same libraries used by the proven single-root notebook, plus pandas for the final status table.

In [3]:
import gc
import sys
import time
import traceback
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyvista as pv
from caveclient import CAVEclient
from cloudvolume import CloudVolume
from meshparty import trimesh_io

import trimesh


def discover_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers."
    )


project_root = discover_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
mesh_dir = project_root / "data" / "meshes"
dec_mesh_dir = mesh_dir / "dec"

if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token
from path_behavior import format_path, print_path

mesh_dir.mkdir(parents=True, exist_ok=True)
dec_mesh_dir.mkdir(parents=True, exist_ok=True)

print_path("mesh_dir", mesh_dir, project_root, show_full_path)
print_path("dec_mesh_dir", dec_mesh_dir, project_root, show_full_path)


mesh_dir: data\meshes
dec_mesh_dir: data\meshes\dec


## Shared Pyr Segmentation Setup

Initialize the public Pyr `seg_m195` MeshParty source once as the primary source. The CAVE-advertised Graphene MeshParty source is initialized lazily only if a flat-source missing-manifest fallback is needed.


In [4]:
def make_mesh_meta_for_source(source):
    mm_candidate = trimesh_io.MeshMeta(
        cv_path=source,
        disk_cache_path=None,
        cache_size=1,
        map_gs_to_https=True,
    )
    cv = mm_candidate.cv
    info = cv.info
    mesh_source = cv.mesh
    return mm_candidate, info, mesh_source


def make_graphene_mesh_meta_for_source(source):
    return trimesh_io.MeshMeta(
        cv_path=source,
        disk_cache_path=None,
        cache_size=1,
    )


candidate_sources = [segmentation_source_gs, segmentation_source_https]
access_errors = []

mm = None
segmentation_source = None
segmentation_info = None

for candidate_source in candidate_sources:
    try:
        mm_candidate, info_candidate, mesh_source_candidate = make_mesh_meta_for_source(candidate_source)
        mm = mm_candidate
        segmentation_source = candidate_source
        segmentation_info = info_candidate
        print("Segmentation metadata access succeeded.")
        print(f"final segmentation source: {segmentation_source}")
        print(f"mesh source type: {type(mesh_source_candidate).__name__}")
        print(f"info type: {segmentation_info.get('type')}")
        print(f"mesh entry: {segmentation_info.get('mesh')}")
        break
    except Exception as exc:
        access_errors.append((candidate_source, type(exc).__name__, str(exc)))
        print(f"Segmentation metadata access failed for {candidate_source}")
        print(f"{type(exc).__name__}: {exc}")

if mm is None:
    raise RuntimeError(f"Could not access public seg_m195 metadata with tested sources: {access_errors}")

for forbidden in ["graphene://", "middleauth+", "minnie.microns-daf.com"]:
    assert forbidden not in segmentation_source.lower()
assert "seg_m195" in segmentation_source


graphene_source = None
mm_graphene = None


def get_graphene_mesh_meta():
    global graphene_source, mm_graphene
    if mm_graphene is not None:
        return mm_graphene

    cave_token, cave_token_source = load_cave_token(project_root)

    if cave_token:
        cave_client = CAVEclient("zheng_ca3", auth_token=cave_token)
    else:
        cave_client = CAVEclient("zheng_ca3")

    graphene_source = cave_client.info.segmentation_source(format_for="cloudvolume")
    if not isinstance(graphene_source, str) or not graphene_source.startswith("graphene://"):
        raise RuntimeError(f"CAVE returned a non-Graphene segmentation source: {graphene_source}")

    mm_graphene = make_graphene_mesh_meta_for_source(graphene_source)
    print("Graphene metadata access succeeded.")
    print(f"Graphene segmentation source: {graphene_source}")
    print(f"Graphene mesh source type: {type(mm_graphene.cv.mesh).__name__}")
    return mm_graphene


Segmentation metadata access succeeded.
final segmentation source: precomputed://gs://zheng_mouse_hippocampus_production/v2/seg_m195/
mesh source type: ShardedMultiLevelPrecomputedMeshSource
info type: segmentation
mesh entry: mesh_mip_1_err_40


## Helper Functions

These helpers keep each root ID isolated while preserving the proven filename conventions, H5 writer/reader, and PyVista triangular-face conversion.

In [5]:
def paths_for_root_id(root_id):
    root_id_str = str(root_id)
    h5_path = mesh_dir / f"mesh_{root_id_str}_mat{materialization_version}.h5"
    ply_path = dec_mesh_dir / f"mesh_{root_id_str}_mat{materialization_version}_dec{dec_prcnt}.ply"
    return h5_path, ply_path


def validate_mesh_arrays(vertices, faces):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces)
    assert vertices.ndim == 2 and vertices.shape[1] == 3
    assert faces.ndim == 2 and faces.shape[1] == 3
    assert len(vertices) > 0
    assert len(faces) > 0
    return vertices, faces


def read_h5_file(file_path):
    with h5py.File(file_path, "r") as f:
        vertices = np.array(f["vertices"])
        faces = np.array(f["faces"])
    return validate_mesh_arrays(vertices, faces)


def create_pyvista_mesh(vertices, faces):
    face_sizes = np.full((faces.shape[0], 1), 3, dtype=faces.dtype)
    faces_with_size = np.hstack([face_sizes, faces]).ravel()
    return pv.PolyData(vertices, faces_with_size)


def validate_pyvista_mesh(mesh):
    assert mesh.n_points > 0
    assert mesh.n_cells > 0
    points = np.asarray(mesh.points)
    assert points.ndim == 2 and points.shape[1] == 3
    return mesh


def temp_path_for(file_path):
    return file_path.with_name(f"{file_path.stem}.tmp{file_path.suffix}")


def remove_if_exists(file_path):
    if file_path.exists():
        file_path.unlink()


def write_h5_atomic(final_path, vertices, faces, mesh):
    temp_path = temp_path_for(final_path)
    try:
        remove_if_exists(temp_path)
        trimesh_io.write_mesh_h5(
            temp_path,
            vertices=vertices,
            faces=faces,
            normals=getattr(mesh, "face_normals", None),
            link_edges=getattr(mesh, "link_edges", None),
            node_mask=getattr(mesh, "node_mask", None),
            overwrite=True,
        )
        assert temp_path.exists()
        reloaded_vertices, reloaded_faces = read_h5_file(temp_path)
        assert len(reloaded_vertices) == len(vertices)
        assert len(reloaded_faces) == len(faces)
        temp_path.replace(final_path)
        assert final_path.exists()
    except Exception:
        remove_if_exists(temp_path)
        raise


def save_ply_atomic(final_path, mesh):
    temp_path = temp_path_for(final_path)
    try:
        remove_if_exists(temp_path)
        mesh.save(temp_path)
        assert temp_path.exists()
        reloaded_mesh = validate_pyvista_mesh(pv.read(temp_path))
        assert reloaded_mesh.n_points == mesh.n_points
        assert reloaded_mesh.n_cells == mesh.n_cells
        temp_path.replace(final_path)
        assert final_path.exists()
        return reloaded_mesh
    except Exception:
        remove_if_exists(temp_path)
        raise


def is_manifest_not_found_error(exc):
    return type(exc).__name__ == "MeshDecodeError" and "Manifest not found" in str(exc)


def classify_exception(exc):
    error_type = type(exc).__name__
    error_message = str(exc)
    if is_manifest_not_found_error(exc):
        return "missing_manifest", error_type, error_message
    return "failed", error_type, error_message


## Per-Root Unit Of Work

Process one root ID at a time. A failure for one root ID is captured in that root's status record and does not stop later root IDs.

In [6]:
def process_root_id(root_id):
    root_id_str = str(root_id)
    h5_path, ply_path = paths_for_root_id(root_id_str)
    start_time = time.time()
    mesh = None
    vertices = None
    faces = None
    reloaded_decimated_mesh = None
    pv_mesh = None
    decimated_mesh = None

    result = {
        "root_id": root_id_str,
        "status": None,
        "materialization_version": materialization_version,
        "mesh_source": None,
        "h5_status": None,
        "ply_status": None,
        "original_vertices": None,
        "original_faces": None,
        "decimated_vertices": None,
        "decimated_faces": None,
        "elapsed_seconds": None,
        "h5_path": str(h5_path),
        "ply_path": str(ply_path),
        "flat_error_type": "",
        "flat_error_message": "",
        "graphene_error_type": "",
        "graphene_error_message": "",
        "error_type": "",
        "error_message": "",
        "error": "",
    }

    try:
        print(f"\nProcessing root ID: {root_id_str}")

        if ply_path.exists() and not overwrite_existing:
            try:
                reloaded_decimated_mesh = validate_pyvista_mesh(pv.read(ply_path))
                result["decimated_vertices"] = reloaded_decimated_mesh.n_points
                result["decimated_faces"] = reloaded_decimated_mesh.n_cells
                result["status"] = "success"
                result["mesh_source"] = "existing_ply"
                result["h5_status"] = "skipped valid PLY"
                result["ply_status"] = "reused existing PLY"
                print(f"Valid PLY already exists; skipping root ID {root_id_str}: {format_path(ply_path, project_root, show_full_path)}")
                return result
            except Exception as exc:
                print(f"Warning: invalid existing PLY for root ID {root_id_str}; removing {format_path(ply_path, project_root, show_full_path)}: {type(exc).__name__}: {exc}")
                ply_path.unlink()

        if h5_path.exists() and not overwrite_existing:
            try:
                print(f"Reusing existing H5: {format_path(h5_path, project_root, show_full_path)}")
                vertices, faces = read_h5_file(h5_path)
                result["mesh_source"] = "existing_h5"
                result["h5_status"] = "reused existing H5"
            except Exception as exc:
                print(f"Warning: invalid existing H5 for root ID {root_id_str}; removing {format_path(h5_path, project_root, show_full_path)}: {type(exc).__name__}: {exc}")
                h5_path.unlink()
                result["h5_status"] = None

        if result["h5_status"] != "reused existing H5":
            try:
                mesh = mm.mesh(
                    seg_id=int(root_id_str),
                    remove_duplicate_vertices=True,
                    cache_mesh=True,
                )
                result["mesh_source"] = "seg_m195"
            except Exception as flat_exc:
                result["flat_error_type"] = type(flat_exc).__name__
                result["flat_error_message"] = str(flat_exc)
                if not is_manifest_not_found_error(flat_exc):
                    raise

                print(
                    f"Flat seg_m195 manifest not found for root ID {root_id_str}; "
                    "trying Graphene fallback"
                )
                try:
                    graphene_mm = get_graphene_mesh_meta()
                    mesh = graphene_mm.mesh(
                        seg_id=int(root_id_str),
                        remove_duplicate_vertices=True,
                    )
                    result["mesh_source"] = "graphene_fallback"
                    print(f"Graphene fallback recovered root ID {root_id_str}")
                except Exception as graphene_exc:
                    result["status"] = "missing_manifest"
                    result["mesh_source"] = "unavailable"
                    result["graphene_error_type"] = type(graphene_exc).__name__
                    result["graphene_error_message"] = str(graphene_exc)
                    result["error_type"] = type(graphene_exc).__name__
                    result["error_message"] = str(graphene_exc)
                    result["error"] = (
                        f"Flat MeshDecodeError: {result['flat_error_message']}; "
                        f"Graphene {type(graphene_exc).__name__}: {graphene_exc}"
                    )
                    result["h5_status"] = result["h5_status"] or "failed"
                    result["ply_status"] = result["ply_status"] or "failed"
                    print(f"Failed root ID {root_id_str}: {result['error']}")
                    return result

            vertices, faces = validate_mesh_arrays(mesh.vertices, mesh.faces)

            write_h5_atomic(h5_path, vertices, faces, mesh)
            result["h5_status"] = "downloaded"
            print(f"Downloaded and saved H5: {format_path(h5_path, project_root, show_full_path)}")

        result["original_vertices"] = len(vertices)
        result["original_faces"] = len(faces)

        pv_mesh = create_pyvista_mesh(vertices, faces)
        assert pv_mesh.n_points == len(vertices)
        assert pv_mesh.n_cells == len(faces)

        target_reduction = dec_prcnt / 100
        decimated_mesh = validate_pyvista_mesh(pv_mesh.decimate(target_reduction=target_reduction))
        result["decimated_vertices"] = decimated_mesh.n_points
        result["decimated_faces"] = decimated_mesh.n_cells

        reloaded_decimated_mesh = save_ply_atomic(ply_path, decimated_mesh)
        assert reloaded_decimated_mesh.n_points == result["decimated_vertices"]
        assert reloaded_decimated_mesh.n_cells == result["decimated_faces"]
        result["status"] = "success"
        result["ply_status"] = "decimated"
        print(f"Decimated and saved PLY: {format_path(ply_path, project_root, show_full_path)}")

    except Exception as exc:
        status, error_type, error_message = classify_exception(exc)
        result["status"] = status
        result["mesh_source"] = result["mesh_source"] or ("unavailable" if status == "missing_manifest" else "failed")
        result["error_type"] = error_type
        result["error_message"] = error_message
        result["error"] = f"{error_type}: {error_message}"
        result["h5_status"] = result["h5_status"] or "failed"
        result["ply_status"] = result["ply_status"] or "failed"
        print(f"Failed root ID {root_id_str}: {result['error']}")
    finally:
        mesh = None
        vertices = None
        faces = None
        reloaded_decimated_mesh = None
        pv_mesh = None
        decimated_mesh = None
        gc.collect()
        result["elapsed_seconds"] = time.time() - start_time

    return result

In [7]:
import psutil
import os

process = psutil.Process(os.getpid())

print(f"Process RAM: {process.memory_info().rss / 1024**3:.2f} GB")
print(f"System available RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")

Process RAM: 0.36 GB
System available RAM: 3.01 GB


## Run Batch

Process each root ID independently and collect one structured status record per root.

In [8]:
results = []
results_csv_path = mesh_dir / "mesh_batch_results.csv"

for root_id in root_ids:
    results.append(process_root_id(root_id))
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_csv_path, index=False)
    print(f"Saved current results table: {format_path(results_csv_path, project_root, show_full_path)}")

results_df = pd.DataFrame(results)
results_df.to_csv(results_csv_path, index=False)
results_display_df = results_df.copy()
for path_column in ["h5_path", "ply_path"]:
    if path_column in results_display_df.columns:
        results_display_df[path_column] = results_display_df[path_column].map(
            lambda path: format_path(path, project_root, show_full_path)
        )
display(results_display_df)


Processing root ID: 648518346458354357


100%|██████████| 1/1 [00:04<00:00,  4.82s/it]


Downloaded and saved H5: data\meshes\mesh_648518346458354357_mat195.h5
Decimated and saved PLY: data\meshes\dec\mesh_648518346458354357_mat195_dec95.ply
Saved current results table: data\meshes\mesh_batch_results.csv


,root_id,status,materialization_version,mesh_source,h5_status,ply_status,original_vertices,original_faces,decimated_vertices,decimated_faces,elapsed_seconds,h5_path,ply_path,flat_error_type,flat_error_message,graphene_error_type,graphene_error_message,error_type,error_message,error
0,648518346458354357,success,195,seg_m195,downloaded,decimated,639986,1271465,32048,63572,17.871969,data\meshes\mesh_648518346458354357_mat195.h5,data\meshes\dec\mesh_648518346458354357_mat195...,,,,,,,


## Batch Summary

Summarize requested, successful, failed, reused, downloaded, and newly decimated roots.

In [9]:
def final_ply_is_valid(root_id):
    _, ply_path = paths_for_root_id(root_id)
    if not ply_path.exists():
        return False
    try:
        validate_pyvista_mesh(pv.read(ply_path))
        return True
    except Exception:
        return False


def print_python_int_list(name, values):
    print(f"{name} = {[int(value) for value in values]}")


successful_mesh_ids = [int(root_id) for root_id in root_ids if final_ply_is_valid(root_id)]
missing_manifest_ids = [
    int(root_id)
    for root_id in results_df.loc[results_df["status"].eq("missing_manifest"), "root_id"]
]
successful_mesh_id_set = set(successful_mesh_ids)
missing_manifest_id_set = set(missing_manifest_ids)
failed_mesh_ids = [
    int(root_id)
    for root_id in root_ids
    if int(root_id) not in successful_mesh_id_set
    and int(root_id) not in missing_manifest_id_set
]

total_requested = len(root_ids)
failed = len(missing_manifest_ids) + len(failed_mesh_ids)
successful = len(successful_mesh_ids)
reused_existing_files = int(
    (results_df["h5_status"].eq("reused existing H5")).sum()
    + (results_df["ply_status"].eq("reused existing PLY")).sum()
)
newly_downloaded = int(results_df["h5_status"].eq("downloaded").sum())
newly_decimated = int(results_df["ply_status"].eq("decimated").sum())
if "mesh_source" not in results_df.columns:
    results_df["mesh_source"] = ""
newly_retrieved_via_seg_m195 = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("seg_m195")).sum()
)
recovered_via_graphene_fallback = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("graphene_fallback")).sum()
)
reused_existing_ply = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("existing_ply")).sum()
)
reused_existing_h5 = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("existing_h5")).sum()
)
remaining_missing_unavailable = len(missing_manifest_ids)
other_failures = len(failed_mesh_ids)

print(f"requested IDs: {total_requested}")
print(f"successful valid PLYs: {successful}")
print(f"missing manifests after fallback: {len(missing_manifest_ids)}")
print(f"other failures: {len(failed_mesh_ids)}")
print(f"reused existing files: {reused_existing_files}")
print(f"newly downloaded: {newly_downloaded}")
print(f"newly decimated: {newly_decimated}")
print()
print("source recovery summary:")
print(f"newly retrieved via seg_m195: {newly_retrieved_via_seg_m195}")
print(f"recovered via Graphene fallback: {recovered_via_graphene_fallback}")
print(f"reused existing PLY: {reused_existing_ply}")
print(f"reused existing H5: {reused_existing_h5}")
print(f"remaining unavailable/failures: {remaining_missing_unavailable + other_failures}")
print(f"remaining missing/unavailable: {remaining_missing_unavailable}")
print(f"other failures: {other_failures}")
print()
print_python_int_list("successful_mesh_ids", successful_mesh_ids)
print()
print_python_int_list("missing_manifest_ids", missing_manifest_ids)
print()
print_python_int_list("failed_mesh_ids", failed_mesh_ids)


requested IDs: 1
successful valid PLYs: 1
missing manifests after fallback: 0
other failures: 0
reused existing files: 0
newly downloaded: 1
newly decimated: 1

source recovery summary:
newly retrieved via seg_m195: 1
recovered via Graphene fallback: 0
reused existing PLY: 0
reused existing H5: 0
remaining unavailable/failures: 0
remaining missing/unavailable: 0
other failures: 0

successful_mesh_ids = [648518346458354357]

missing_manifest_ids = []

failed_mesh_ids = []
